In [ ]:
from pathlib import Path
import ast
import re

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

CONSTRUTOR = ROOT / "exp/eft_tiny_512_v1/eft_modelo_672.py"
ALP = ROOT / "models/grid_proto_fewshot.py"
REFERENCIA_672 = (
    ROOT / "exp/eft_validacao_polipos_v1/"
    "validar_referencia_atual.py"
)

PILOTO = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
)

assert CONSTRUTOR.is_file()
assert ALP.is_file()
assert REFERENCIA_672.is_file()
assert (PILOTO / "checkpoint_0000.pt").is_file()


def mostrar_funcao(arquivo, nome, limite=180):
    texto = arquivo.read_text(encoding="utf-8")
    linhas = texto.splitlines()
    arvore = ast.parse(texto)

    encontrados = [
        no for no in ast.walk(arvore)
        if isinstance(
            no, (ast.FunctionDef, ast.AsyncFunctionDef)
        )
        and no.name == nome
    ]

    print(f"\n=== {arquivo.relative_to(ROOT)} ===")
    print("Função:", nome)
    print("Ocorrências:", len(encontrados))

    for no in encontrados[:2]:
        inicio = no.lineno
        fim = no.end_lineno

        print(f"Linhas {inicio}–{fim}")

        for numero in range(inicio, min(fim, inicio + limite - 1) + 1):
            print(f"{numero:04d}: {linhas[numero-1]}")

        if fim - inicio + 1 > limite:
            print("... função truncada para inspeção")


# 1. Como construir um VMamba em outra resolução?
mostrar_funcao(
    CONSTRUTOR,
    "construir_modelo",
    limite=180
)

# 2. Como o ALP trabalha com os mapas de características?
mostrar_funcao(
    ALP,
    "get_features",
    limite=55
)

# 3. Localizar configurações de resolução já utilizadas
# em experimentos VMamba anteriores.
print("\n=== REFERÊNCIA VMAMBA 672 ANTERIOR ===")

linhas = REFERENCIA_672.read_text(
    encoding="utf-8"
).splitlines()

padrao = re.compile(
    r"image_size|feature_hw|FewShotSeg|"
    r"vmamba|checkpoint|pretrained_path",
    re.IGNORECASE
)

indices = [
    i for i, linha in enumerate(linhas)
    if padrao.search(linha)
]

mostrar = set()

for i in indices[:35]:
    mostrar.update(
        range(
            max(0, i - 1),
            min(len(linhas), i + 3)
        )
    )

for i in sorted(mostrar):
    print(f"{i+1:04d}: {linhas[i]}")

# 4. Conferir os checkpoints disponíveis para
# iniciar a avaliação sem qualquer treinamento.
print("\n=== CHECKPOINT MRI INICIAL ===")

cp = PILOTO / "checkpoint_0000.pt"

print("Arquivo:", cp)
print("Disponível:", cp.is_file())
print("Tamanho:", round(cp.stat().st_size / 1024**2, 1), "MiB")

print("\nINSPEÇÃO CONCLUÍDA — nenhum arquivo modificado.")

In [ ]:
from pathlib import Path
import sys
import csv
import gc
import importlib.util

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT_ALP = (
    ROOT / "exp/eft_mri_chaos_v1/"
    "avaliar_alp_mri_10000.py"
)

CHECKPOINT = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200/"
    "checkpoint_0000.pt"
)

REF_256 = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

assert SCRIPT_ALP.is_file()
assert CHECKPOINT.is_file()
assert REF_256.is_file()
assert torch.cuda.is_available()

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False

# Reutilizar exatamente o avaliador MRI anterior.
spec = importlib.util.spec_from_file_location(
    "avaliador_vmamba_mri_256", SCRIPT_ALP
)
base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)

print("=" * 65)
print("VMAMBA-TINY CONGELADO — MRI 256 versus 672")
print("=" * 65)

# Construir o modelo e restaurar o checkpoint inicial.
modelo = base.construir_modelo("tiny", repo=ROOT)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

assert int(checkpoint["passo"]) == 0
modelo.load_state_dict(checkpoint["modelo"], strict=True)
del checkpoint

modelo.eval().requires_grad_(False)

# Mesmos pacientes e fatias dos diagnósticos anteriores.
classe = 2  # rim direito

imgs_s, labels_s = base.ler_volume(4)
imgs_q, labels_q = base.ler_volume(0)

positivas_s = np.flatnonzero(
    np.any(labels_s == classe, axis=(1, 2))
)
positivas_q = np.flatnonzero(
    np.any(labels_q == classe, axis=(1, 2))
)

zs = int(positivas_s[len(positivas_s) // 2])
zq = int(positivas_q[len(positivas_q) // 2])

support_256 = base.preparar_imagem(imgs_s[zs])
query_256 = base.preparar_imagem(imgs_q[zq])

fg_256 = torch.from_numpy(
    (labels_s[zs] == classe).astype(np.float32)
)[None].to("cuda")

gt = labels_q[zq] == classe

assert support_256.shape == (1, 3, 256, 256)
assert query_256.shape == (1, 3, 256, 256)
assert fg_256.shape == (1, 256, 256)

# Construir a versão 672:
# imagem bilinear, máscara nearest-neighbor.
support_672 = F.interpolate(
    support_256,
    size=(672, 672),
    mode="bilinear",
    align_corners=False
)

query_672 = F.interpolate(
    query_256,
    size=(672, 672),
    mode="bilinear",
    align_corners=False
)

fg_672 = F.interpolate(
    fg_256[:, None],
    size=(672, 672),
    mode="nearest"
)[:, 0]

assert fg_672.shape == (1, 672, 672)
assert torch.all(
    (fg_672 == 0) | (fg_672 == 1)
).item()

print(f"Suporte: paciente 4, fatia {zs}")
print(f"Query: paciente 0, fatia {zq}")
print("Checkpoint: 0 — sem EFT")

# Inspecionar a resolução realmente entregue pelo encoder.
capturas = []

def registrar_encoder(modulo, entrada, saida):
    tensor = (
        saida[-1]
        if isinstance(saida, (list, tuple))
        else saida
    )
    capturas.append(tuple(tensor.shape))

hook = modelo.encoder.register_forward_hook(
    registrar_encoder
)

def avaliar(support, fg, query, nome):
    capturas.clear()

    with torch.inference_mode():
        retorno = modelo(
            [[support]],
            [[fg]],
            [[1 - fg]],
            [query],
            isval=True,
            val_wsize=2,
            show_viz=False
        )

        logits = retorno[0]

        assert torch.isfinite(logits).all().item()
        assert logits.shape[-2:] == query.shape[-2:]

        # Métricas sempre calculadas em 256 × 256.
        if logits.shape[-2:] != (256, 256):
            logits_metricas = F.interpolate(
                logits,
                size=(256, 256),
                mode="bilinear",
                align_corners=False
            )
        else:
            logits_metricas = logits

        pred = (
            logits_metricas.argmax(dim=1)[0]
            .cpu().numpy() == 1
        )

    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())

    dice = 2 * tp / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)

    print(f"\n=== {nome} ===")
    print("Entrada:", tuple(query.shape))
    print("Características nativas:", capturas)
    print("Logits:", tuple(logits.shape))
    print(f"Dice: {dice:.6f}")
    print(f"IoU: {iou:.6f}")
    print("Pixels previstos:", int(pred.sum()))

    return dice, iou

try:
    dice_256, iou_256 = avaliar(
        support_256, fg_256, query_256, "VMamba 256"
    )

    dice_672, iou_672 = avaliar(
        support_672, fg_672, query_672, "VMamba 672"
    )

    # Confirmar que a execução 256 é igual à anterior.
    with REF_256.open(
        newline="", encoding="utf-8"
    ) as f:
        referencia = [
            r for r in csv.DictReader(f)
            if int(r["checkpoint"]) == 0
            and r["orgao"] == "RK"
            and int(r["paciente"]) == 0
            and int(r["fatia"]) == zq
        ]

    assert len(referencia) == 1

    dice_salvo = float(referencia[0]["dice"])

    print("\n=== VERIFICAÇÃO DO BASELINE ===")
    print(f"Dice 256 salvo:       {dice_salvo:.6f}")
    print(f"Dice 256 reproduzido: {dice_256:.6f}")

    assert np.isclose(
        dice_salvo,
        dice_256,
        atol=1e-6,
        rtol=0
    ), "O baseline VMamba 256 não foi reproduzido."

    print("\n=== COMPARAÇÃO FINAL ===")
    print(f"Dice VMamba 256: {dice_256:.4f}")
    print(f"Dice VMamba 672: {dice_672:.4f}")
    print(
        f"Diferença: {(dice_672-dice_256)*100:+.2f} p.p."
    )

    print("\nDIAGNÓSTICO CONCLUÍDO")

finally:
    hook.remove()
    del modelo
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
import importlib.util
import gc

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = (
    ROOT / "exp/eft_mri_chaos_v1/"
    "avaliar_alp_mri_10000.py"
)

CHECKPOINT = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200/"
    "checkpoint_0000.pt"
)

assert SCRIPT.is_file()
assert CHECKPOINT.is_file()
assert torch.cuda.is_available()

spec = importlib.util.spec_from_file_location(
    "vmamba_mri_inspecao", SCRIPT
)

base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)

torch.backends.cuda.matmul.allow_tf32 = False

modelo = None
hooks = []

def formatos(obj):
    if isinstance(obj, torch.Tensor):
        return tuple(obj.shape)
    if isinstance(obj, (tuple, list)):
        return [formatos(x) for x in obj]
    if isinstance(obj, dict):
        return {k: formatos(v) for k, v in obj.items()}
    return type(obj).__name__

try:
    modelo = base.construir_modelo("tiny", repo=ROOT)

    checkpoint = torch.load(
        CHECKPOINT,
        map_location="cpu",
        weights_only=False
    )

    assert int(checkpoint["passo"]) == 0

    modelo.load_state_dict(
        checkpoint["modelo"],
        strict=True
    )

    del checkpoint

    modelo.eval().requires_grad_(False)
    encoder = modelo.encoder

    print("=" * 68)
    print("INSPEÇÃO DAS CAMADAS DO VMAMBA-TINY — CHAOS MRI")
    print("=" * 68)

    print("Checkpoint: 0 — pesos iniciais")
    print("Treinamento habilitado: NÃO")
    print("Configuração ALP:", modelo.config["feature_hw"])

    print("\n=== ESTÁGIOS DISPONÍVEIS ===")

    assert hasattr(encoder, "layers")

    for indice, camada in enumerate(encoder.layers):
        print(
            f"Stage {indice}: "
            f"{type(camada).__name__}"
        )

    print("\n=== NORMALIZAÇÕES DISPONÍVEIS ===")

    for nome, modulo in encoder.named_modules():
        if (
            "outnorm" in nome.lower()
            or "classifier.norm" in nome.lower()
        ):
            print(nome, type(modulo).__name__)

    print("\n=== EXTRAÇÃO DE CARACTERÍSTICAS ===")

    capturas = {}

    def registrar(nome):
        def hook(modulo, entrada, saida):
            capturas[nome] = formatos(saida)
        return hook

    for indice, camada in enumerate(encoder.layers):
        hooks.append(
            camada.register_forward_hook(
                registrar(f"stage_{indice}")
            )
        )

    # Mesmo suporte utilizado nos diagnósticos anteriores.
    imagens, labels = base.ler_volume(4)

    positivas = np.flatnonzero(
        np.any(labels == 2, axis=(1, 2))
    )

    z = int(positivas[len(positivas) // 2])

    entrada = base.preparar_imagem(
        imagens[z]
    ).to("cuda")

    entrada_672 = F.interpolate(
        entrada,
        size=(672, 672),
        mode="bilinear",
        align_corners=False
    )

    with torch.inference_mode():
        saida_encoder = encoder(entrada_672)

    print("Paciente: 4")
    print("Fatia:", z)
    print("Entrada:", tuple(entrada_672.shape))

    for nome, shape in capturas.items():
        print(f"{nome}: {shape}")

    print(
        "Saída atual do encoder:",
        formatos(saida_encoder)
    )

    print("\n=== ÍNDICES DE SAÍDA ===")

    print(
        "out_indices:",
        getattr(encoder, "out_indices", "não encontrado")
    )

    print("\nINSPEÇÃO CONCLUÍDA")

finally:
    for hook in hooks:
        hook.remove()

    if modelo is not None:
        del modelo

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
ARQUIVO = ROOT / "models/vmamba.py"

assert ARQUIVO.is_file()

fonte = ARQUIVO.read_text(encoding="utf-8")
linhas = fonte.splitlines()
arvore = ast.parse(fonte)

print("=" * 72)
print("INSPEÇÃO DO FORWARD — VMAMBA")
print("=" * 72)

classes = {
    no.name: no
    for no in arvore.body
    if isinstance(no, ast.ClassDef)
}

print("\n=== CLASSES RELEVANTES ===")

for nome in sorted(classes):
    if any(
        termo in nome.lower()
        for termo in ("vssm", "vssblock", "backbone")
    ):
        print(nome)

def mostrar_metodo(classe, metodo, limite=110):
    no = classes.get(classe)

    if no is None:
        print(f"\nClasse {classe} não encontrada.")
        return

    encontrados = [
        n for n in no.body
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
        and n.name == metodo
    ]

    if not encontrados:
        print(f"\n{classe}.{metodo}: não definido nesta classe")
        return

    for funcao in encontrados:
        print(f"\n=== {classe}.{metodo} ===")
        print(f"Linhas {funcao.lineno}–{funcao.end_lineno}")

        fim = min(
            funcao.end_lineno,
            funcao.lineno + limite - 1
        )

        for numero in range(funcao.lineno, fim + 1):
            print(f"{numero:04d}: {linhas[numero - 1]}")

        if fim < funcao.end_lineno:
            print("... restante omitido")

# Inspecionar como o backbone efetivamente
# percorre os estágios.
mostrar_metodo("Backbone_VSSM", "forward", 130)
mostrar_metodo("VSSM", "forward", 100)
mostrar_metodo("VSSM", "forward_features", 100)

# Identificar como os blocos são organizados.
mostrar_metodo("Backbone_VSSM", "__init__", 115)

# Localizar definições das normalizações e
# referências às saídas intermediárias.
print("\n=== REFERÊNCIAS A ESTÁGIOS E NORMALIZAÇÕES ===")

termos = (
    "out_indices",
    "outnorm",
    "self.layers",
    "self.downsamples",
    "self.patch_embed",
)

indices = [
    i for i, linha in enumerate(linhas)
    if any(termo in linha for termo in termos)
]

# Limitar a saída, mantendo o contexto.
for i in indices[:65]:
    print(f"{i+1:04d}: {linhas[i].rstrip()}")

print("\nINSPEÇÃO CONCLUÍDA")

In [ ]:
from pathlib import Path
import importlib.util
import csv
import gc
import types

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

SCRIPT = ROOT / "exp/eft_mri_chaos_v1/avaliar_alp_mri_10000.py"

CHECKPOINT = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_piloto200"
    / "checkpoint_0000.pt"
)

REFERENCIA = (
    ROOT / "reproducao/eft_mri_chaost2_fold0_10000_v1"
    / "avaliacao_alp_rins_positivas_f0_7ckpts"
    / "metricas_fatias.csv"
)

assert SCRIPT.is_file()
assert CHECKPOINT.is_file()
assert REFERENCIA.is_file()
assert torch.cuda.is_available()

spec = importlib.util.spec_from_file_location(
    "mri_stage2_diagnostico", SCRIPT
)
base = importlib.util.module_from_spec(spec)
spec.loader.exec_module(base)

from models.alpmodule import MultiProtoAsConv

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False

modelo = None
hook = None

try:
    # 1. Restaurar o baseline inicial.
    modelo = base.construir_modelo("tiny", repo=ROOT)

    cp = torch.load(
        CHECKPOINT, map_location="cpu", weights_only=False
    )

    assert int(cp["passo"]) == 0
    modelo.load_state_dict(cp["modelo"], strict=True)
    del cp

    modelo.eval().requires_grad_(False)

    assert modelo.config["feature_hw"] == [32, 32]
    assert modelo.encoder.out_indices == (3,)
    assert modelo.encoder.channel_first

    # 2. Recuperar mesmo suporte e query.
    classe = 2

    imgs_s, labels_s = base.ler_volume(4)
    imgs_q, labels_q = base.ler_volume(0)

    fatias_s = np.flatnonzero(
        np.any(labels_s == classe, axis=(1, 2))
    )
    fatias_q = np.flatnonzero(
        np.any(labels_q == classe, axis=(1, 2))
    )

    zs = int(fatias_s[len(fatias_s) // 2])
    zq = int(fatias_q[len(fatias_q) // 2])

    support = base.preparar_imagem(imgs_s[zs])
    query = base.preparar_imagem(imgs_q[zq])

    fg = torch.from_numpy(
        (labels_s[zs] == classe).astype(np.float32)
    )[None].to("cuda")

    gt = labels_q[zq] == classe

    print("=" * 68)
    print("CHAOS MRI — VMAMBA-TINY: STAGE 2 VERSUS STAGE 3")
    print("=" * 68)
    print("Checkpoint: 0 (sem EFT)")
    print(f"Suporte: paciente 4, fatia {zs}")
    print(f"Query: paciente 0, fatia {zq}")
    print("Grade ALP: 32 × 32")

    def avaliar(nome):
        with torch.inference_mode():
            retorno = modelo(
                [[support]],
                [[fg]],
                [[1 - fg]],
                [query],
                isval=True,
                val_wsize=2,
                show_viz=False
            )

            logits = retorno[0]

            assert logits.shape == (1, 2, 256, 256)
            assert torch.isfinite(logits).all().item()

            pred = (
                logits.argmax(dim=1)[0]
                .cpu().numpy() == 1
            )

        tp = int((pred & gt).sum())
        fp = int((pred & ~gt).sum())
        fn = int((~pred & gt).sum())

        dice = 2 * tp / max(2 * tp + fp + fn, 1)
        iou = tp / max(tp + fp + fn, 1)

        print(f"\n=== {nome} ===")
        print(f"Dice: {dice:.6f}")
        print(f"IoU:  {iou:.6f}")
        print("Pixels previstos:", int(pred.sum()))

        return dice

    # 3. Executar última camada, sem modificações.
    dice_stage3 = avaliar("Stage 3 — baseline original")

    with REFERENCIA.open(
        newline="", encoding="utf-8"
    ) as f:
        casos = [
            r for r in csv.DictReader(f)
            if int(r["checkpoint"]) == 0
            and r["orgao"] == "RK"
            and int(r["paciente"]) == 0
            and int(r["fatia"]) == zq
        ]

    assert len(casos) == 1
    dice_salvo = float(casos[0]["dice"])

    print(f"Baseline salvo: {dice_salvo:.6f}")

    assert np.isclose(
        dice_stage3, dice_salvo, rtol=0, atol=1e-6
    ), "Stage 3 não reproduziu o baseline."

    # 4. Capturar saída do Stage 2 antes do downsampling.
    capturas = {}

    def capturar_stage2(modulo, entrada, saida):
        assert isinstance(saida, torch.Tensor)
        capturas["stage2"] = saida

    hook = modelo.encoder.layers[2].blocks.register_forward_hook(
        capturar_stage2
    )

    # A normalização do Stage 2 é experimental:
    # não existem pesos outnorm2 no checkpoint.
    estado = {"normalizar": False}
    formas = []

    def extrair_stage2(self, imgs_concat):
        imagens = F.interpolate(
            imgs_concat,
            size=(672, 672),
            mode="bilinear",
            align_corners=False
        )

        capturas.clear()

        # Executar o encoder original.
        # O hook captura a saída dos blocos do Stage 2.
        _ = self.encoder(imagens)

        assert "stage2" in capturas, (
            "O hook do Stage 2 não foi acionado."
        )

        features = capturas.pop("stage2")

        assert features.ndim == 4
        assert features.shape[1:] == (384, 42, 42), (
            f"Formato Stage 2 inesperado: {features.shape}"
        )

        formas.append(tuple(features.shape))

        if estado["normalizar"]:
            # LayerNorm por posição espacial, sobre 384 canais.
            # Sem pesos aprendidos (elementwise_affine=False).
            features = F.layer_norm(
                features.permute(0, 2, 3, 1),
                normalized_shape=(384,)
            ).permute(0, 3, 1, 2).contiguous()

        return F.interpolate(
            features,
            size=(32, 32),
            mode="bilinear",
            align_corners=False
        )

    # 5. Substituir somente a origem dos features e
    # ajustar o ALP para 384 canais.
    modelo.get_features = types.MethodType(
        extrair_stage2, modelo
    )

    modelo.cls_unit = MultiProtoAsConv(
        proto_grid=[8, 8],
        feature_hw=[32, 32],
        embed_dim=384
    ).to("cuda")

    modelo.eval().requires_grad_(False)

    # A comparação não deve introduzir pesos treináveis.
    assert not any(
        p.requires_grad for p in modelo.parameters()
    )

    estado["normalizar"] = False
    dice_stage2_raw = avaliar("Stage 2 — sem normalização")

    estado["normalizar"] = True
    dice_stage2_ln = avaliar(
        "Stage 2 — LayerNorm sem parâmetros aprendidos"
    )

    print("\n=== GEOMETRIA CAPTURADA ===")
    for forma in sorted(set(formas)):
        print("Stage 2:", forma)

    assert formas
    assert all(
        forma[1:] == (384, 42, 42)
        for forma in formas
    )

    print("\n=== COMPARAÇÃO ===")
    print(f"Stage 3 original:       {dice_stage3:.4f}")
    print(f"Stage 2 bruto:          {dice_stage2_raw:.4f}")
    print(f"Stage 2 + LayerNorm:    {dice_stage2_ln:.4f}")

    print(
        "Δ Stage 2 bruto:       "
        f"{(dice_stage2_raw-dice_stage3)*100:+.2f} p.p."
    )
    print(
        "Δ Stage 2 + LayerNorm: "
        f"{(dice_stage2_ln-dice_stage3)*100:+.2f} p.p."
    )

    print("\nDIAGNÓSTICO CONCLUÍDO")

finally:
    if hook is not None:
        hook.remove()

    if modelo is not None:
        del modelo

    gc.collect()
    torch.cuda.empty_cache()